# Fine-tuning Qwen2.5-VL-3B-Instruct (versión optimizada con r=16 + early stopping)

Cambios respecto al primer run (`r=64, alpha=64, 3 epochs fijas`):
- **`r = 16, alpha = 32`** (factor 2.0): rank más bajo aprovechando que el modelo base es de 3B (heurística "menos rank con modelo grande, alpha proporcionalmente mayor").
- **Early stopping con patience=1**: para de entrenar si `eval_loss` no mejora en 1 epoch. En el run anterior viste que epoch 3 ya no aportaba (mejora de 0.00008). Ahorra cómputo.
- **`num_train_epochs = 5`**: tope alto, pero early stopping lo cortará antes.
- **`model_short = "qwen25_vl_3b_r16"`**: nombre distinto para no sobrescribir el JSON del primer run.

Resultados se guardan en `../results/qwen25_vl_3b_r16.json`.

## 1. Setup

In [1]:
import os
import json
import time
from pathlib import Path
from collections import defaultdict

from dotenv import load_dotenv
load_dotenv()

import comet_ml

assert os.environ.get("HF_TOKEN", "").startswith("hf_"), "Falta HF_TOKEN en .env"
assert os.environ.get("COMET_API_KEY"), "Falta COMET_API_KEY en .env"
print("Tokens cargados.")

Tokens cargados.


## 2. Descargar dataset (idempotente)

In [2]:
from huggingface_hub import snapshot_download

HF_DATASET = "damianGil/wildfire-prevention"

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data" / "wildfire"
RESULTS_DIR  = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=HF_DATASET,
    repo_type="dataset",
    local_dir=str(DATA_DIR),
)
print(f"Dataset listo en {DATA_DIR}")

/home/damian/unsloth-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching ... files: 2084it [00:08, 246.97it/s]

Dataset listo en /mnt/c/Users/Usuario/Documents/GitHub/finetunning-library/code/wildfire-prevention/data/wildfire


## 3. Cargar modelo Qwen2.5-VL-3B

In [3]:
from unsloth import FastVisionModel
import torch

MODEL_NAME  = "unsloth/Qwen2.5-VL-3B-Instruct"
MODEL_SHORT = "qwen25_vl_3b_r16"

model, tokenizer = FastVisionModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 4096,
    load_in_4bit   = False,
)

print(f"Modelo cargado: {MODEL_NAME}")
print(f"VRAM reservada: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 12.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights: 100%|██████████| 824/824 [00:14<00:00, 55.37it/s] 


Modelo cargado: unsloth/Qwen2.5-VL-3B-Instruct
VRAM reservada: 7.07 GB


## 4. Configurar LoRA (r=16, alpha=32 — factor 2.0)

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r            = 16,
    lora_alpha   = 32,
    lora_dropout = 0.05,
    bias         = "none",
    random_state = 42,
    use_rslora   = False,
    loftq_config = None,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


trainable params: 41,084,928 || all params: 3,795,707,904 || trainable%: 1.0824


## 5. Preparar dataset

In [5]:
from datasets import load_dataset
from PIL import Image

ds = load_dataset(str(DATA_DIR))
print(f"Train: {len(ds['train'])}  |  Test: {len(ds['test'])}")

SYSTEM_PROMPT = """\
You are a remote sensing analyst specialising in wildfire risk assessment.
You will be given two Sentinel-2 satellite images of the same land tile:
  1. RGB composite (bands B4-B3-B2): natural colour, useful for terrain, infrastructure, and land cover.
  2. SWIR composite (bands B12-B8-B4): highlights vegetation moisture stress and dryness. Healthy vegetation appears green/cyan, stressed or dry vegetation appears orange/red, bare soil appears magenta/pink, and burned areas appear dark red or black.

Assess the wildfire risk of the tile and return ONLY a valid JSON object — no markdown, no explanation outside the JSON — with exactly these fields:

{
  "risk_level": "low | medium | high",
  "dry_vegetation_present": true | false,
  "urban_interface": true | false,
  "steep_terrain": true | false,
  "water_body_present": true | false,
  "image_quality_limited": true | false
}
"""

USER_TEXT = (
    "Image 1 is the RGB composite. Image 2 is the SWIR composite. "
    "Return the wildfire risk JSON for this tile."
)
PROMPT = f"{SYSTEM_PROMPT.strip()}\n\n{USER_TEXT}"


def to_unsloth_conversation(sample):
    rgb  = Image.open(DATA_DIR / sample["rgb_path"]).convert("RGB")
    swir = Image.open(DATA_DIR / sample["swir_path"]).convert("RGB")
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": rgb},
                {"type": "image", "image": swir},
                {"type": "text",  "text":  PROMPT},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": sample["output"]},
            ]},
        ]
    }


train_dataset = [to_unsloth_conversation(s) for s in ds["train"]]
test_dataset  = [to_unsloth_conversation(s) for s in ds["test"]]
print(f"Conversaciones preparadas. Train={len(train_dataset)}, Test={len(test_dataset)}")

Train: 862  |  Test: 172
Conversaciones preparadas. Train=862, Test=172


## 6. Trainer + entrenamiento (con early stopping)

In [6]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,
    callbacks     = [EarlyStoppingCallback(early_stopping_patience=1)],
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        num_train_epochs            = 5,    # tope alto, early stopping lo cortará
        warmup_ratio                = 0.03,
        learning_rate               = 1e-4,
        lr_scheduler_type           = "cosine",
        logging_steps               = 5,
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        seed                        = 42,
        output_dir                  = f"outputs/{MODEL_SHORT}",
        run_name                    = f"{MODEL_SHORT}-r16-alpha32-vision-on-early",
        report_to                   = ["comet_ml"],

        eval_strategy               = "epoch",
        per_device_eval_batch_size  = 4,
        save_strategy               = "epoch",
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,

        remove_unused_columns = False,
        dataset_text_field    = "",
        dataset_kwargs        = {"skip_prepare_dataset": True},
        max_length            = 4096,
    ),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512


In [7]:
torch.cuda.reset_peak_memory_stats()
start_mem = torch.cuda.memory_reserved() / 1024**3
print(f"VRAM antes de train: {start_mem:.2f} GB")

trainer_stats = trainer.train()

peak_mem = torch.cuda.max_memory_reserved() / 1024**3
actual_epochs = trainer_stats.metrics.get('epoch', 0)
print(f"\nTraining: {trainer_stats.metrics['train_runtime']/60:.2f} min")
print(f"Epochs reales (early stopping): {actual_epochs:.1f}")
print(f"VRAM pico: {peak_mem:.2f} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.


VRAM antes de train: 7.24 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 862 | Num Epochs = 5 | Total steps = 540
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/damian-gil-gonzalez/wildfire-finetune/62498dc9ddef40cc8c28cb6cd14cfb6a

COMET WARNING: String value length exceeds 1000 characters and will be truncated. Provided value: 'LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Qwen2_5_VLForConditionalGeneration', 'parent_library': 'transformers.models.qwen2_5_vl.modeling_q

Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.005962,0.008010
2,0.005244,0.006761
3,0.003458,0.005309
4,0.002643,0.005343


Unsloth: Restored added_tokens_decoder metadata in outputs/qwen25_vl_3b_r16/checkpoint-108/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/qwen25_vl_3b_r16/checkpoint-216/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/qwen25_vl_3b_r16/checkpoint-324/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/qwen25_vl_3b_r16/checkpoint-432/tokenizer_config.json.



Training: 115.44 min
Epochs reales (early stopping): 4.0
VRAM pico: 14.08 GB


## 7. Evaluación completa sobre test

In [8]:
FastVisionModel.for_inference(model)

FIELDS = ["risk_level", "dry_vegetation_present", "urban_interface",
          "steep_terrain", "water_body_present", "image_quality_limited"]

N_EVAL = len(ds["test"])
correct = defaultdict(int)
valid_json_count = 0
inf_times = []

for i in range(N_EVAL):
    s = ds["test"][i]
    rgb  = Image.open(DATA_DIR / s["rgb_path"]).convert("RGB")
    swir = Image.open(DATA_DIR / s["swir_path"]).convert("RGB")
    gt   = json.loads(s["output"])

    msgs = [{"role": "user", "content": [
        {"type": "image"}, {"type": "image"}, {"type": "text", "text": PROMPT}
    ]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inp  = tokenizer([rgb, swir], text, add_special_tokens=False, return_tensors="pt").to("cuda")

    t0 = time.time()
    out_ids = model.generate(**inp, max_new_tokens=256, use_cache=True,
                              temperature=0.1, min_p=0.1)
    inf_times.append(time.time() - t0)

    pred_text = tokenizer.decode(out_ids[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        clean = pred_text.strip()
        if clean.startswith("```"):
            clean = clean.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        pred = json.loads(clean)
        valid_json_count += 1
        for f in FIELDS:
            if pred.get(f) == gt.get(f):
                correct[f] += 1
    except (json.JSONDecodeError, IndexError):
        pass

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{N_EVAL} muestras evaluadas")

valid_json    = valid_json_count / N_EVAL
accuracy_per_field = {f: correct[f] / N_EVAL for f in FIELDS}
overall = sum(correct.values()) / (N_EVAL * len(FIELDS))
avg_inf_time = sum(inf_times) / len(inf_times)

print(f"\n=== RESULTADOS Qwen2.5-VL-3B (r=16, alpha=32, early stop) ===")
print(f"valid_json:  {valid_json:.3f}")
for f in FIELDS:
    print(f"  {f:25s} {accuracy_per_field[f]:.3f}")
print(f"\noverall: {overall:.3f}")
print(f"tiempo medio inferencia/muestra: {avg_inf_time:.2f} s")

Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  20/172 muestras evaluadas


Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  40/172 muestras evaluadas


Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  140/172 muestras evaluadas


Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  160/172 muestras evaluadas


Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


=== RESULTADOS Qwen2.5-VL-3B (r=16, alpha=32, early stop) ===
valid_json:  1.000
  risk_level                0.820
  dry_vegetation_present    0.860
  urban_interface           0.965
  steep_terrain             0.860
  water_body_present        0.884
  image_quality_limited     0.924

overall: 0.886
tiempo medio inferencia/muestra: 2.50 s


## 8. Guardar resultados

In [9]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

results = {
    "model_name":  MODEL_NAME,
    "model_short": MODEL_SHORT,
    "training": {
        "num_params_total":      total_params,
        "num_params_trainable":  trainable_params,
        "vram_peak_gb":          peak_mem,
        "training_time_min":     trainer_stats.metrics['train_runtime'] / 60,
        "epochs_actual":         trainer_stats.metrics.get('epoch', None),
        "loss_train_final":      trainer_stats.training_loss,
        "config": {
            "r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
            "finetune_vision_layers": True,
            "load_in_4bit":           False,
            "num_train_epochs_max":   5,
            "early_stopping_patience": 1,
            "batch_effective":        8,
            "learning_rate":          1e-4,
            "lr_scheduler":           "cosine",
            "warmup_ratio":           0.03,
        },
    },
    "eval": {
        "num_samples":           N_EVAL,
        "valid_json":            valid_json,
        "accuracy_per_field":    accuracy_per_field,
        "overall":               overall,
        "inference_time_per_sample_s": avg_inf_time,
    },
}

out_path = RESULTS_DIR / f"{MODEL_SHORT}.json"
out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Resultados guardados en: {out_path}")
print(json.dumps(results, indent=2))

Resultados guardados en: /mnt/c/Users/Usuario/Documents/GitHub/finetunning-library/code/wildfire-prevention/results/qwen25_vl_3b_r16.json
{
  "model_name": "unsloth/Qwen2.5-VL-3B-Instruct",
  "model_short": "qwen25_vl_3b_r16",
  "training": {
    "num_params_total": 3795707904,
    "num_params_trainable": 41084928,
    "vram_peak_gb": 14.080078125,
    "training_time_min": 115.43834166666667,
    "epochs_actual": 4.0,
    "loss_train_final": 0.12452984135813529,
    "config": {
      "r": 16,
      "lora_alpha": 32,
      "lora_dropout": 0.05,
      "finetune_vision_layers": true,
      "load_in_4bit": false,
      "num_train_epochs_max": 5,
      "early_stopping_patience": 1,
      "batch_effective": 8,
      "learning_rate": 0.0001,
      "lr_scheduler": "cosine",
      "warmup_ratio": 0.03
    }
  },
  "eval": {
    "num_samples": 172,
    "valid_json": 1.0,
    "accuracy_per_field": {
      "risk_level": 0.8197674418604651,
      "dry_vegetation_present": 0.8604651162790697,
      